In [1]:
import logging
from transformers import logging as tf_logging

tf_logging.set_verbosity_error()

In [2]:
import sys

In [3]:
import asyncio
import json
import os
import time
from pathlib import Path

import dotenv
from tqdm import tqdm

from financial_qa.chunkers.semantic import SemanticChunker
from financial_qa.chunkers.table_split import TableSplitChunker
from financial_qa.embedders import GigaEmbedder
from financial_qa.rag import RAG
from financial_qa.agent.agent_loop import OpenRouterAgentLoop, _clean_path, _query_log_path
from financial_qa.evaluation import evaluate_async, load_jsonl

In [4]:
dotenv.load_dotenv('.env')

GIGACHAT_CREDENTIALS = os.getenv('GIGACHAT_CREDENTIALS')
if not GIGACHAT_CREDENTIALS:
    raise ValueError('GIGACHAT_CREDENTIALS is required (used for embeddings)')

OPENROUTER_API_KEY = os.getenv('OPENROUTER_API_KEY')
if not OPENROUTER_API_KEY:
    raise ValueError('OPENROUTER_API_KEY is required')

DATASET_FILE = 'dataset.jsonl'
DATASET_SPLIT = None
MAX_QUESTIONS = 50

RAG_DB = 'html_table_split_semantic_giga_embeddings_mh4_c6000'
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200
MAX_ROWS_PER_CHUNK = 10
MAX_CHUNK_CHARS = 6000
MIN_CHUNK_LEN = 0
TOP_K = 10
EMBED_MODEL = 'EmbeddingsGigaR'
GIGACHAT_SCOPE = 'GIGACHAT_API_PERS'

# google/gemma-4-31b-it — verify the exact model slug on openrouter.ai/models
GEN_MODEL = 'google/gemma-4-26b-a4b-it'
QUERY_CONCURRENCY = 5

MAX_TURNS = 8
MIN_RETRIEVALS_BEFORE_NOT_FOUND = 3

JUDGE_MODEL = 'google/gemini-2.0-flash-lite-001'
JUDGE_PROCESSES = 100

In [5]:
all_records = load_jsonl(DATASET_FILE)

In [6]:
len(all_records)

449

In [7]:
all_records = load_jsonl(DATASET_FILE)

all_records = [
    all_records[r]
    for r in all_records
    if DATASET_SPLIT is None or all_records[r].get('split') == DATASET_SPLIT
]

seen = set()
records = []
cnt = 0
for r in all_records:
    if r['question_id'] not in seen:
        records.append(r)
        seen.add(r['question_id'])
        cnt += 1
print(cnt)

if MAX_QUESTIONS:
    records = records[:MAX_QUESTIONS]

golden = {r['question_id']: r for r in records}
print(f'Loaded {len(records)} records (split={DATASET_SPLIT!r})')
print('Sample:', json.dumps(records[0], ensure_ascii=False, indent=2))

449
Loaded 50 records (split=None)
Sample: {
  "question_id": "q_00d660efcf3e4607",
  "question": "Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?",
  "split": "test",
  "gold_evidence": [
    {
      "doc_id": "alfa_2025_annual",
      "pages": [
        103
      ]
    }
  ],
  "gold_answer": "1,151 тыс. белорусских рублей"
}


## Patched agent loop (v2 + v3 fixes) — OpenRouter / Gemma 4 31B

Same fixes as the GigaChat v2/v3 patch, ported to the OpenRouter `tool_calls` protocol.

| Fix | Where | What changed |
|-----|-------|-------------|
| Prevent repeated queries | prompt + loop | `seen_retrieve_calls` guard |
| Force minimum 3 retrievals | loop | Block "not found" if < `MIN_RETRIEVALS_BEFORE_NOT_FOUND` unique calls; inject retry hint |
| No-digit chunk detection | loop | Append `retrieval_hint` to tool result when no chunk contains a number |
| 3-tier fallback query strategy | prompt | Full query → short keyword → section/note name |
| Sign rules for reserves | prompt | ECL reserves always positive; balance ≠ movement |
| Explicit source citation for ratios | prompt | Must quote both numbers verbatim before calculating |
| Partial enumeration answers | prompt | List all elements when asked |
| Accounting parentheses | prompt | `(3 685)` → `3 685` |
| calculate tool | loop + prompt | Safe arithmetic eval via sandboxed AST |
| OpenRouter tool_calls format | loop | Uses `tool_calls`/`role:tool` instead of GigaChat `function_call`/`role:function` |

In [8]:
import ast as _ast
import re as _re
from typing import Any, Optional

_NOT_FOUND_PHRASES = [
    "не найдено", "не была найдена", "не удалось найти", "информация не найдена",
    "нет информации", "нет необходимой", "отсутствует информация", "не обнаружена",
    "не удалось обнаружить", "недостаточно информации", "не найден", "не найдены",
    "не содержит", "не была непосредственно", "не было найдено", "отсутствует в",
    "информация отсутствует", "не была обнаружена",
]


def _looks_like_not_found(text: str) -> bool:
    t = text.lower()
    return any(p in t for p in _NOT_FOUND_PHRASES)


def _has_digit(text: str) -> bool:
    return bool(_re.search(r"\d", text))


class PatchedOpenRouterAgentLoop(OpenRouterAgentLoop):
    """OpenRouterAgentLoop with improved Russian-language prompt and loop-level retry enforcement."""

    def _tool_spec(self) -> list[dict[str, Any]]:
        tools = super()._tool_spec()
        tools.append(
            {
                "type": "function",
                "function": {
                    "name": "calculate",
                    "description": (
                        "Safely evaluate a numeric arithmetic expression. "
                        "Input: {expression: string}. Supports parentheses, unary +/-, and operators +, -, *, /. "
                        "Returns {result: number}. Does not allow variables, functions, exponentiation, or modulo."
                    ),
                    "parameters": {
                        "type": "object",
                        "properties": {
                            "expression": {"type": "string"},
                        },
                        "required": ["expression"],
                    },
                },
            }
        )
        return tools

    def _safe_eval_expression(self, expression: str) -> float:
        expr = expression.strip()
        if not expr:
            raise ValueError("Empty expression")
        node = _ast.parse(expr, mode="eval")

        def _eval(n: _ast.AST) -> float:
            if isinstance(n, _ast.Expression):
                return _eval(n.body)
            if isinstance(n, _ast.Constant) and isinstance(n.value, (int, float)):
                return float(n.value)
            if isinstance(n, _ast.UnaryOp) and isinstance(n.op, (_ast.UAdd, _ast.USub)):
                value = _eval(n.operand)
                return value if isinstance(n.op, _ast.UAdd) else -value
            if isinstance(n, _ast.BinOp) and isinstance(n.op, (_ast.Add, _ast.Sub, _ast.Mult, _ast.Div)):
                left = _eval(n.left)
                right = _eval(n.right)
                if isinstance(n.op, _ast.Add): return left + right
                if isinstance(n.op, _ast.Sub): return left - right
                if isinstance(n.op, _ast.Mult): return left * right
                if isinstance(n.op, _ast.Div):
                    if right == 0:
                        raise ZeroDivisionError("Division by zero")
                    return left / right
            raise ValueError(f"Unsupported node: {type(n).__name__}")

        return _eval(node)

    def _build_messages(self, query_str: str) -> list[dict[str, Any]]:
        structure = self.rag.get_structure()
        system_prompt = (
            "Ты — ассистент для ответов на вопросы по финансовым отчётам банков. "
            "Перед тем как ответить, ОБЯЗАТЕЛЬНО вызови функцию retrieve_file_chunks "
            "для получения нужных фрагментов из документов. "
            "Для любых арифметических вычислений (суммы, разницы, проценты, доли) "
            "используй функцию calculate. "
            "Интерфейс calculate: вход — {expression: строка}, выход — {result: число}. "
            "Поддерживает скобки, унарные +/-, и операторы +, -, *, /. "
            "Не поддерживает переменные, функции, возведение в степень или модуль. "
            "Используй каталог файлов ниже, чтобы выбрать подходящий файл. "
            "Можешь вызывать функцию несколько раз для разных файлов.\n\n"
            "Правила извлечения данных:\n"
            "- Для вопросов о доле, соотношении или проценте (слова «доля», «отношение», «процент», «во сколько раз») "
            "ВСЕГДА делай два отдельных вызова retrieve_file_chunks: "
            "первый — с запросом на числитель, второй — с запросом на знаменатель. "
            "В запросах указывай точную сущность, дату и строку таблицы.\n"
            "- НИКОГДА не повторяй тот же самый query дважды для одного и того же файла. "
            "Если первый запрос вернул нерелевантный фрагмент — переформулируй запрос.\n"
            "- Стратегия поиска при неудаче (применяй последовательно):\n"
            "    Шаг 1 — точная формулировка из вопроса + дата\n"
            "    Шаг 2 — только название строки/статьи + дата (2–4 слова), "
            "например: «резервы по гарантиям 30 июня 2024» или «государственные облигации 2025»\n"
            "    Шаг 3 — название раздела/примечания, "
            "например: «операции со связанными сторонами», «Департамент рисков», «реформа процентных ставок»\n"
            "- После получения фрагментов проверь: содержит ли фрагмент конкретное числовое значение, "
            "отвечающее на вопрос? Если нет (или в нём только заголовки и текст без цифр) — "
            "вызови retrieve_file_chunks ещё раз с другим, более коротким запросом.\n\n"
            "Правила ответа:\n"
            "- Финальный ответ возвращай СТРОГО в формате JSON (без markdown-обёртки):\n"
            "  {\"answer\": \"...\", \"confidence\": 0.0}\n"
            "- answer — итоговое число, дату или факт. "
            "Для вопросов «сколько», «какие», «перечисли» — перечисли все элементы полностью.\n"
            "- Для вопросов о доле/проценте: ПЕРЕД расчётом явно процитируй оба числа из фрагментов "
            "и название строки таблицы, откуда каждое взято. "
            "Пример: «числитель — 10 835 млн руб. (строка «оценочный резерв»), "
            "знаменатель — 36 508 млн руб. (строка «итого прочие фин. активы»), доля = 29,7%». "
            "Если не можешь найти оба числа в полученных фрагментах — сделай ещё один запрос. "
            "НИКОГДА не изобретай числа.\n"
            "- Резервы под ОКУ (ECL) — всегда положительные числа в ответе. "
            "Если в таблице резерв показан в скобках или со знаком минус — "
            "это бухгалтерская запись: в ответе пиши положительное значение.\n"
            "- Вопрос «резерв/остаток НА дату X» требует остатка (balance) на эту дату, "
            "НЕ движения за период (не изменение, не начисление за период).\n"
            "- confidence — число от 0.0 до 1.0, основанное на поле score фрагментов.\n"
            "  Если нужный факт явно присутствует в топ-фрагменте (score > 0.85), "
            "ставь confidence близко к score.\n"
            "  Если пришлось собирать из слабых фрагментов — снижай confidence.\n"
            "  Если информация не найдена — confidence = 0.0.\n"
            "- НИКОГДА не пиши 'необходимо', 'следует', 'для расчёта нужно' и т.п.\n"
            "- Опирайся только на данные из полученных фрагментов."
        )
        user_prompt = (
            f"Вопрос:\n{query_str}\n\n"
            "Каталог доступных файлов отчётов:\n"
            f"{structure}"
        )
        return [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

    async def aquery(self, query_str: str, query_id: Optional[str] = None) -> tuple[str, Optional[float]]:
        import hashlib

        self._last_confidence = None
        query_id = query_id or hashlib.md5(query_str.encode("utf-8")).hexdigest()
        log_path = _query_log_path(self.log_root, query_id) if self.log_root else None
        self._log_event(
            log_path, "query_start",
            {"query_id": query_id, "question": query_str, "model": self.model},
        )
        messages = self._build_messages(query_str)
        tools = self._tool_spec()
        used_scores: list[float] = []
        seen_retrieve_calls: set[tuple[str, str]] = set()

        for turn in range(1, self.max_turns + 1):
            if turn == self.max_turns:
                messages.append({
                    "role": "user",
                    "content": (
                        "Это последний шаг. НЕ вызывай функции. "
                        'Дай финальный ответ строго в формате JSON: {"answer": "...", "confidence": 0.0}. '
                        "Никакого текста вне JSON."
                    ),
                })

            self._log_event(log_path, "llm_request", {
                "query_id": query_id, "turn": turn,
                "message_count": len(messages), "tool_count": len(tools),
                "messages": messages, "tools": tools,
            })
            body = await self._call_llm(messages=messages, tools=tools)
            choice = body["choices"][0]
            message = choice["message"]
            tool_calls = message.get("tool_calls") or []

            self._log_event(log_path, "llm_response", {
                "query_id": query_id, "turn": turn,
                "tool_calls": len(tool_calls),
                "has_content": bool(message.get("content")),
                "response": body,
            })

            if tool_calls:
                messages.append({
                    "role": "assistant",
                    "content": message.get("content") or "",
                    "tool_calls": tool_calls,
                })

                for call in tool_calls:
                    fn = call["function"]["name"]
                    args = json.loads(call["function"]["arguments"])
                    call_id = call["id"]

                    if fn == "retrieve_file_chunks":
                        path = _clean_path(args.get("path", ""))
                        sub_query = args.get("query", query_str)
                        call_key = (path, sub_query)

                        if call_key in seen_retrieve_calls:
                            correction = (
                                f"Ты уже делал этот запрос (path='{path}', query='{sub_query}'). "
                                "Попробуй другой запрос: сократи до ключевого названия строки таблицы и даты, "
                                "или поищи в другом файле."
                            )
                            self._log_event(log_path, "repeated_query_blocked",
                                            {"query_id": query_id, "path": path, "query": sub_query})
                            messages.append({"role": "tool", "tool_call_id": call_id, "name": fn,
                                             "content": json.dumps({"error": correction}, ensure_ascii=False)})
                            continue

                        seen_retrieve_calls.add(call_key)
                        try:
                            self._log_event(log_path, "tool_call", {
                                "query_id": query_id, "tool": fn,
                                "path": path, "query": sub_query, "tool_call": call,
                            })
                            results = await self.rag.aretrieve(path, sub_query)
                            chunks = [{"text": c.text, "doc": c.doc, "pos": c.pos, "score": s}
                                      for c, s in results]
                            scores = [c["score"] for c in chunks]
                            tool_payload = {"path": path, "query": sub_query, "chunks": chunks}

                            if chunks and not any(_has_digit(c["text"]) for c in chunks):
                                tool_payload["retrieval_hint"] = (
                                    "ВНИМАНИЕ: ни один из полученных фрагментов не содержит числовых данных. "
                                    "Попробуй более короткий запрос: только название строки/статьи + дата."
                                )
                                self._log_event(log_path, "no_digit_chunks",
                                                {"query_id": query_id, "path": path, "query": sub_query})

                            self._log_event(log_path, "tool_result", {
                                "query_id": query_id, "tool": fn, "path": path,
                                "chunks": len(chunks),
                                "max_score": max(scores) if scores else None,
                                "result": tool_payload,
                            })
                            used_scores.extend(scores)
                            messages.append({"role": "tool", "tool_call_id": call_id, "name": fn,
                                             "content": json.dumps(tool_payload, ensure_ascii=False)})
                        except Exception as e:
                            self._log_event(log_path, "tool_error",
                                            {"query_id": query_id, "tool": fn, "tool_call": call, "error": str(e)})
                            messages.append({"role": "tool", "tool_call_id": call_id, "name": fn,
                                             "content": json.dumps({"error": str(e)}, ensure_ascii=False)})

                    elif fn == "calculate":
                        expression = args.get("expression", "")
                        try:
                            self._log_event(log_path, "tool_call", {
                                "query_id": query_id, "tool": fn,
                                "expression": expression, "tool_call": call,
                            })
                            result_val = self._safe_eval_expression(expression)
                            tool_payload = {"expression": expression, "result": result_val}
                            self._log_event(log_path, "tool_result",
                                            {"query_id": query_id, "tool": fn, "result": tool_payload})
                            messages.append({"role": "tool", "tool_call_id": call_id, "name": fn,
                                             "content": json.dumps(tool_payload, ensure_ascii=False)})
                        except Exception as e:
                            self._log_event(log_path, "tool_error",
                                            {"query_id": query_id, "tool": fn, "tool_call": call, "error": str(e)})
                            messages.append({"role": "tool", "tool_call_id": call_id, "name": fn,
                                             "content": json.dumps({"error": str(e)}, ensure_ascii=False)})

                    else:
                        messages.append({"role": "tool", "tool_call_id": call_id, "name": fn,
                                         "content": json.dumps({"error": f"Unknown function: {fn}"}, ensure_ascii=False)})

                continue

            # ── model wants to give final answer ───────────────────────────────
            raw = (message.get("content") or "").strip()
            answer, model_confidence = self._parse_final_answer(raw)

            n_unique = len(seen_retrieve_calls)
            if (
                _looks_like_not_found(answer)
                and n_unique < MIN_RETRIEVALS_BEFORE_NOT_FOUND
                and turn < self.max_turns - 1
            ):
                self._log_event(log_path, "not_found_retry_forced",
                                {"query_id": query_id, "turn": turn,
                                 "unique_retrievals": n_unique, "answer": answer})
                messages.append({"role": "assistant", "content": raw})
                messages.append({
                    "role": "user",
                    "content": (
                        f"Ты заявил «не найдено», но сделал только {n_unique} уникальных запрос(а). "
                        f"ОБЯЗАТЕЛЬНО попробуй ещё минимум {MIN_RETRIEVALS_BEFORE_NOT_FOUND - n_unique} запрос(а) "
                        "с другими ключевыми словами:\n"
                        "- Сократи до 2–3 слов: только название статьи/раздела + дата\n"
                        "- Для связанных сторон: «операции со связанными сторонами [тип]»\n"
                        "- Для подразделений: попробуй прямое название (например «Департамент рисков»)\n"
                        "- Для IBOR/реформа: «реформа процентных ставок», «альтернативные базовые ставки»"
                    ),
                })
                continue

            if model_confidence is not None:
                self._last_confidence = model_confidence
            elif used_scores:
                self._last_confidence = max(used_scores)

            self._log_event(log_path, "query_complete", {
                "query_id": query_id, "answer": answer,
                "confidence": self._last_confidence, "messages": messages,
            })
            return answer, self._last_confidence

        self._log_event(log_path, "query_failed",
                        {"query_id": query_id, "reason": "max_turns_exceeded", "messages": messages})
        return "Unable to complete tool-calling loop within max_turns.", self._last_confidence

In [9]:
text_chunker = SemanticChunker(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
chunker = TableSplitChunker(text_chunker=text_chunker, max_rows_per_chunk=MAX_ROWS_PER_CHUNK, max_chunk_chars=MAX_CHUNK_CHARS)
embedder = GigaEmbedder(
    credentials=GIGACHAT_CREDENTIALS,
    model=EMBED_MODEL,
    scope=GIGACHAT_SCOPE,
)
rag = RAG(
    chunker=chunker,
    embedder=embedder,
    data_dir='data/parsed',
    store_dir='indexes',
    name=RAG_DB,
    top_k=TOP_K,
    min_chunk_len=MIN_CHUNK_LEN,
)

store_dir = Path('indexes') / RAG_DB
has_index = store_dir.exists() and any(store_dir.glob('*.npz'))
if not has_index:
    print('No index found; running precalc...')
    rag.precalc(max_workers=1)
else:
    print(f'Using existing index at {store_dir}')

loop = PatchedOpenRouterAgentLoop(
    rag=rag,
    model=GEN_MODEL,
    max_turns=MAX_TURNS,
)

Using existing index at indexes/html_table_split_semantic_giga_embeddings_mh4_c6000


In [10]:
async def run_queries(records):
    predictions = {}
    errors = []
    timings = []
    semaphore = asyncio.Semaphore(QUERY_CONCURRENCY)

    async def _query_one(rec):
        start = time.perf_counter()
        try:
            answer, confidence = await loop.aquery(rec['question'])
            error = None
        except Exception as e:
            answer = ''
            confidence = None
            error = str(e)
        elapsed = time.perf_counter() - start
        return {
            'question_id': rec['question_id'],
            'question': rec['question'],
            'answer': answer,
            'evidence': [],
            'confidence': confidence,
            'error': error,
            'elapsed_s': elapsed,
        }

    async def _bound(rec):
        async with semaphore:
            return await _query_one(rec)

    tasks = {asyncio.create_task(_bound(rec)): rec for rec in records}
    progress = tqdm(total=len(tasks), desc='Querying agent', unit='question')
    for task in asyncio.as_completed(tasks):
        result = await task
        predictions[result['question_id']] = result
        if result['error']:
            errors.append(result)
        timings.append(result['elapsed_s'])
        progress.update(1)
        progress.set_postfix(
            errors=len(errors),
            avg_s=f"{sum(timings)/len(timings):.2f}",
            last_conf=result['confidence'],
        )
    progress.close()
    return predictions, errors

predicted, query_errors = await run_queries(records)
print(f'Done: {len(predicted)} answers, {len(query_errors)} errors')

Querying agent:   0%|          | 0/50 [00:00<?, ?question/s]

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 12s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…


Querying agent:  10%|█         | 5/50 [00:17<01:27,  1.95s/question, avg_s=12.69, errors=0, last_conf=0.81]

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…
[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 11s…


Querying agent:  20%|██        | 10/50 [00:35<01:34,  2.37s/question, avg_s=14.49, errors=0, last_conf=0.82]

...

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 10s…


Querying agent:  74%|███████▍  | 37/50 [01:17<00:17,  1.33s/question, avg_s=9.76, errors=1, last_conf=0.796]

[GigaEmbedder] 429 on embeddings (attempt 1/6), sleeping 10s…


Querying agent: 100%|██████████| 50/50 [01:46<00:00,  2.13s/question, avg_s=10.20, errors=2, last_conf=0.88] 

Done: 50 answers, 2 errors


In [11]:
query_errors

[{'question_id': 'q_1a9dcd8e81499b4d',
  'question': 'Насколько изменились итоговые обязательства кредитного характера за вычетом оценочного резерва под кредитные убытки ЗАО «Альфа-Банк» на 31 декабря 2024 года по сравнению с 31 декабря 2023 года, и каков при этом объём средств клиентов, привлечённых от компаний под общим контролем, на 31 декабря 2024 года?',
  'answer': '',
  'evidence': [],
  'confidence': None,
  'error': 'Server disconnected',
  'elapsed_s': 20.21463124989532},
 {'question_id': 'q_455e24ad587d8689',
  'question': 'Какую долю от общей суммы процентных доходов по финансовым инструментам, переоцениваемым по справедливой стоимости через прибыль или убыток, составляют процентные доходы по инструментам уровня 3 иерархии справедливой стоимости у Совкомбанка за шесть месяцев, завершившихся 30 июня 2025 г.?',
  'answer': '',
  'evidence': [],
  'confidence': None,
  'error': 'Server disconnected',
  'elapsed_s': 5.012895165942609}]

In [12]:
result = await evaluate_async(
    golden=golden,
    predicted=predicted,
    model=JUDGE_MODEL,
    detailed_result=True,
    include_evidence=False,
    use_processes=True,
    max_workers=JUDGE_PROCESSES,
    progress_desc='LLM-as-judge',
)

LLM-as-judge: 100%|██████████| 50/50 [00:01<00:00, 29.59question/s, accuracy=58.00%, correct=29, errors=0]


In [13]:
result['correct'] / result['total']

0.58

In [14]:
accuracy = result["correct"] / result["total"]

conf_scores = []
correct_conf = []
incorrect_conf = []

for res in result["results"]:
    confidence = predicted.get(res["question_id"], {}).get("confidence")
    if confidence is None:
        continue
    score = confidence if res["judge_score"] == 1 else 1 - confidence
    conf_scores.append(score)

    if res["judge_score"] == 1:
        correct_conf.append(confidence)
    else:
        incorrect_conf.append(confidence)

conf_precision = sum(conf_scores) / len(conf_scores) if conf_scores else float("nan")

print(f"Accuracy:                   {accuracy:.2%} ({result['correct']}/{result['total']})")
print(f"Confidence precision:       {conf_precision:.4f}")
print(f"Avg confidence (correct):   {sum(correct_conf) / len(correct_conf):.4f}")
print(f"Avg confidence (incorrect): {sum(incorrect_conf) / len(incorrect_conf):.4f}")
print()
print("Baseline GigaChat v3: 54% | v2: 50% | v1: 44–48%")

Accuracy:                   58.00% (29/50)
Confidence precision:       0.6878
Avg confidence (correct):   0.8342
Avg confidence (incorrect): 0.7839

Baseline GigaChat v3: 54% | v2: 50% | v1: 44–48%


In [15]:
for res in result['results'][0:10]:
    print(res)
    print('-' * 75)

{'question_id': 'q_00d660efcf3e4607', 'question': 'Какова общая сумма финансовых обязательств ЗАО «Альфа-Банк», подлежащих переводу на альтернативные процентные базовые ставки, по состоянию на 31 декабря 2025 года?', 'gold_answer': '1,151 тыс. белорусских рублей', 'predicted_answer': 'Unable to complete tool-calling loop within max_turns.', 'judge_score': 0, 'judge_reasoning': 'Предсказанный ответ не содержит никакой информации, в то время как золотой ответ содержит конкретное число.'}
---------------------------------------------------------------------------
{'question_id': 'q_017f831a3f0e16a3', 'question': 'Применяет ли ЗАО «Альфа-Банк» учёт хеджирования в отношении производных финансовых инструментов согласно финансовой отчётности за 2022 год?', 'gold_answer': 'Нет, Банк не применяет учёт хеджирования.', 'predicted_answer': 'Нет, согласно финансовой отчётности за 2022 год, Банк не применяет учёт хеджирования в отношении производных финансовых инструментов. Изменения справедливой ст

In [16]:
import shutil
shutil.make_archive("logs", "zip", "logs")

'/Users/kitlix/CProjects/hse/ai360_proj_may_2026/ai360-financial-qa/logs.zip'